# Agent 05 - Validate Local Data vs Ping Range Master

## Funcion

Validar si la descarga local cubre la ventana operativa oficial (`first_day` -> `last_day`) de cada ticker definida por `ping_range_master`.

## Como debemos hacerlo

1. Leer `ping_range_master` (fuente de verdad operativa).
2. Extraer d?as observados locales por ticker desde `quotes.parquet`.
3. Comparar d?as esperados vs observados.
4. Guardar:
   - `coverage_vs_ping_current.csv`
   - `gaps_by_ticker.csv`

Este notebook es para control de completitud; puede ejecutarse cada cierto tiempo durante la descarga.



**Qué hace `agent_05_validate_local_vs_ping.ipynb`**

Ruta:

- C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\00_data_certification\agent_05_validate_local_vs_ping.ipynb

Función real:

1. Lee PING_MASTER_PARQUET.
2. Escanea local QUOTES_ROOT ({ticker}/year=YYYY/month=MM/day=DD/quotes.parquet) para días observados.
3. Compara esperado vs observado por ticker.
4. Saca:
   - cobertura por ticker
   - gaps/faltantes

Nota: hoy usa bdate_range para expected days (aprox. hábiles), no calendario oficial exchange.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# --- Config ---
QUOTES_ROOT = Path(r"C:\TSIS_Data\data\quotes_p95")
PING_MASTER_PARQUET = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\ping_range_master.parquet")

OUT_DIR = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\agent05_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

COVERAGE_OUT_CSV = OUT_DIR / "coverage_vs_ping_current.csv"
GAPS_OUT_CSV = OUT_DIR / "gaps_by_ticker.csv"

MAX_TICKERS = None  # None = todos

print("QUOTES_ROOT:", QUOTES_ROOT)
print("PING_MASTER_PARQUET:", PING_MASTER_PARQUET)


In [ ]:
# --- Cargar ping master ---
if not PING_MASTER_PARQUET.exists():
    raise FileNotFoundError(f"No existe ping master: {PING_MASTER_PARQUET}")

pm = pd.read_parquet(PING_MASTER_PARQUET)
for c in ["ticker", "first_day", "last_day", "has_data"]:
    if c not in pm.columns:
        raise RuntimeError(f"Columna requerida ausente en ping master: {c}")

pm = pm.copy()
pm["ticker"] = pm["ticker"].astype(str).str.strip()
pm = pm[pm["ticker"] != ""]
pm = pm[pm["has_data"].fillna(False)]
pm["first_day"] = pd.to_datetime(pm["first_day"], errors="coerce")
pm["last_day"] = pd.to_datetime(pm["last_day"], errors="coerce")
pm = pm.dropna(subset=["first_day", "last_day"]) 

if MAX_TICKERS not in (None, ""):
    pm = pm.head(int(MAX_TICKERS)).copy()

print("tickers_ping_validos:", len(pm))
display(pm.head(10))


In [ ]:
# --- Extraer dias observados locales por ticker desde estructura de carpetas ---
# Patron esperado: {ticker}/year=YYYY/month=MM/day=DD/quotes.parquet

files = list(QUOTES_ROOT.rglob("quotes.parquet"))
print("quotes_files_detectados:", len(files))

rows = []
for fp in files:
    # ...\quotes_p95\TICKER\year=YYYY\month=MM\day=DD\quotes.parquet
    parts = fp.parts
    try:
        ticker = parts[-5]
        y = parts[-4].replace("year=", "")
        m = parts[-3].replace("month=", "")
        d = parts[-2].replace("day=", "")
        dt = pd.to_datetime(f"{y}-{m}-{d}", errors="coerce")
        if pd.notna(dt):
            rows.append((ticker, dt.normalize()))
    except Exception:
        continue

obs = pd.DataFrame(rows, columns=["ticker", "date"]).drop_duplicates()
obs["ticker"] = obs["ticker"].astype(str).str.strip()

print("observed_ticker_days:", len(obs))
display(obs.head(10))


In [ ]:
# --- Comparar expected vs observed ---
# Nota: se usa bdate_range (dias habiles). Para precision exchange/feriados,
# sustituir por calendario oficial de mercado.

obs_count = obs.groupby("ticker", dropna=False)["date"].nunique().rename("present_days").reset_index()

exp_rows = []
for _, r in pm.iterrows():
    tk = r["ticker"]
    d0 = r["first_day"].normalize()
    d1 = r["last_day"].normalize()
    if d1 < d0:
        continue
    expected_days = len(pd.bdate_range(d0, d1))
    exp_rows.append((tk, d0, d1, expected_days))

exp = pd.DataFrame(exp_rows, columns=["ticker", "exp_min", "exp_max", "expected_days"])
coverage = exp.merge(obs_count, on="ticker", how="left")
coverage["present_days"] = coverage["present_days"].fillna(0).astype(int)
coverage["missing_days"] = (coverage["expected_days"] - coverage["present_days"]).clip(lower=0)
coverage["coverage_ratio"] = np.where(coverage["expected_days"] > 0, coverage["present_days"] / coverage["expected_days"], np.nan)

# gaps detallados por ticker (lista de dias faltantes)
obs_map = obs.groupby("ticker")["date"].apply(lambda s: set(pd.to_datetime(s).normalize())).to_dict()

gaps_rows = []
for _, r in exp.iterrows():
    tk = r["ticker"]
    expected_set = set(pd.bdate_range(r["exp_min"], r["exp_max"]).normalize())
    present_set = obs_map.get(tk, set())
    missing = sorted(expected_set - present_set)
    for d in missing:
        gaps_rows.append((tk, pd.Timestamp(d).date()))

gaps = pd.DataFrame(gaps_rows, columns=["ticker", "missing_day"]) if gaps_rows else pd.DataFrame(columns=["ticker", "missing_day"])

print("coverage_rows:", len(coverage))
print("gaps_rows:", len(gaps))
display(coverage.sort_values("coverage_ratio", ascending=True).head(20))


In [ ]:
# --- Guardar salidas ---
coverage.to_csv(COVERAGE_OUT_CSV, index=False, encoding="utf-8")
gaps.to_csv(GAPS_OUT_CSV, index=False, encoding="utf-8")

print("saved:", COVERAGE_OUT_CSV)
print("saved:", GAPS_OUT_CSV)
